In [ ]:
import requests

url = "https://api.sportradar.com/tennis/trial/v3/en/complexes.json"

headers = {
    "accept": "application/json",
    "x-api-key": "i2oSr6rhWNZa7hsdSZsDINmpVcj5sy1Xahg2sZuB"
}

response = requests.get(url, headers=headers)

if response.status_code == 200:
    complexes_data = response.json()              # Step - 1 : Convert the response into dictionary format
    
    print(complexes_data)
else:
    print("Error in API:", response.status_code)

In [ ]:
complexes_data                     # Step - 2 : View the data and its structure.

In [27]:
# Step - 3 : PostgreSql connection and creating the table.

import psycopg2

connection = psycopg2.connect(
    host = "localhost",
    database = "sports",
    user = "postgres",
    password = "12345678",
    port = "5432" )

cursor = connection.cursor()

cursor.execute('''CREATE TABLE IF NOT EXISTS complexes_table 
               (complex_id VARCHAR(50)PRIMARY KEY,                 
               complex_name VARCHAR(100) NOT NULL)''')


print("complexes table created successfully")


cursor.execute(''' CREATE TABLE IF NOT EXISTS venues_table (
                venue_id VARCHAR(50) PRIMARY KEY,
                venue_name VARCHAR(100) NOT NULL,
                city_name VARCHAR(100) NOT NULL,
                country_name VARCHAR(100) NOT NULL,
                country_code CHAR(3) NOT NULL,
                timezone VARCHAR(100) NOT NULL,
                complex_id VARCHAR(50),
                FOREIGN KEY (complex_id) REFERENCES complexes_table(complex_id))
''')

connection.commit()

print("venues table created successfully")

complexes table created successfully
venues table created successfully


In [28]:
# Step - 3 : Extract the complexes from the given data.

for comp in complexes_data['complexes']:
        
        complex_id = comp['id'].replace('sr:complex:','')
        
        complex_name = comp['name']

# Step - 4 : Inserting the complexes into the database.

        cursor.execute('''
                INSERT INTO complexes_table(complex_id,complex_name)
                VALUES(%s,%s) ON CONFLICT (complex_id) DO NOTHING''',
                (complex_id,complex_name))
        
connection.commit()

print("complexes inserted into the database successfully")
                       

complexes inserted into the database successfully


In [31]:
# Step - 5 : Extract the venues from the given data.

for comp in complexes_data['complexes']:
       
      venues = comp.get('venues')

      if not venues:
           continue

      for venue in comp['venues']:

         venue_id = venue['id'].replace('sr:venue:','')

         venue_name = venue['name']

# Step - 6 : Inserting the venues into the database.

         cursor.execute('''
                  INSERT INTO venues_table(venue_id,venue_name,city_name,
                  country_name,country_code,timezone,complex_id) VALUES
                  (%s,%s,%s,%s,%s,%s,%s) ON CONFLICT (venue_id) DO 
                  NOTHING''',(venue_id,venue_name,venue['city_name'],
                  venue['country_name'],venue['country_code'],venue['timezone'],complex_id))
         

connection.commit()


print("venues inserted into the database successfully")


venues inserted into the database successfully
